# Week 2: Gradient Descent, Cross-Validation, and Classification

This week develops gradient descent for regression, introduces train/dev/test splits for cross-validation, and moves from regression to classification with activation functions and cross-entropy.

**Lecture 3:** Gradient descent and cross-validation  
**Lecture 4:** Classifiers, activations, and cross-entropy


# Lecture 3: Gradient Descent and Cross-Validation


### Least Squares by Gradient Descent

For linear regression, we often need to minimize a loss function such as the squared-error objective below.

$$\min\limits_w\,L(w)=\min\limits_w\,\frac{1}{n}\|Xw-y\|^2$$

We will use the method of **gradient descent** to find an approximate solution. Gradient descent is a very quick method exploiting some pretty simple ideas from multivariable calculus. While it will only find an approximate answer at best, it is practically good enough in most cases. The benefits are that it is computationally cheap and can be used for many other useful loss functions.

Gradient descent (and it's sped-up version, **stochastic gradient descent** or **SGD**) is *heavily* used in machine learning. Along with backpropagation, SGD is the primary method used for training neural networks. In fact, the first practical neural networks were written about in the machine learning literature under a class of methods called "gradient-based learning" due to the primacy of SGD and related methods.

### Some Ideas from Multivariate Calculus

First, we need just a few ideas from multivariate calculus. This is quite minimal, but you learn more details about these topics in sections 4.3 (partial derivatives), 4.6 (gradients), and 4.7 (multivariate optimization) of <a href="https://openstax.org/details/books/calculus-volume-3">*Calculus Volume 3*</a> by Strang.

The ideas we need for gradient descent include:

* If we have a differentiable function of several variables, like our loss function $L(w) = L(w_0, ..., w_d)$, we can define the **partial derivatives** with respect to each of these variables as

    $$L_{w_i}=\frac{\partial L}{\partial w_i}=\lim\limits_{h\to 0}\frac{L(w+he_i) - L(w)}{h},$$

    where $e_i$ is a $(d+1)$-vector with all 0s except for a 1 in the $i$th component.  Geometrically, this partial derivative is the slope of $L$ if we go in the direction of $e_i$.
    
* To **minimize a multivariable function** by hand, we need to find critical points, which are points $w$ where *all* partial derivatives are 0 and compare which ones give the lowest outputs. In numerical algorithms, must settle for approximations that are "nearly" critical points.
    
* If we put these partial derivatives into a vector of $d+1$ variables, we call that a **gradient**, which we denote
    
    $$
    \nabla L(w)
    =\begin{pmatrix}
    L_{w_0}(w) \\
    \vdots \\
    L_{w_d}(w)
    \end{pmatrix}
    $$
    
* The **directional derivative** of a function in the direction of a unit vector $u$ starting from a point $w$.

    $$D_u L(w) = \lim\limits_{h\to 0}\frac{L(w+hu) - L(w)}{h},$$
    
    which is the slope in the direction of the vector $u$. Since $L$ is differentiable, this directional derivative will be defined for all directions leaving from $w$. In 1D, there are just two directions: left of right. In 2D, we have directional derivatives at every angle in a circle around the point.
    
* A common theorem says the **directional derivative is maximized in the direction of the gradient** at each point, so the gradient gives the direction of the *steepest ascent* in the function $L$. Similarly, the direction of the *steepest descent* is $-\nabla L(w)$, the opposite direction.

#### The Geometry of Gradient Descent

We will discuss the geometry of gradient-based methods in class, but let's discuss a general outline of how gradient descent works, setting aside the stochastic version for now. The goal of gradient descent is to approximately solve the minimization problem

$$\min\limits_{w}\,L(w)$$

by finding (approximate) critical values by making a guess for the location of a critical value, taking a small step in the opposite direction as the gradient, and repeating this over and over until, hopefully, we reach a minimum value.

The steps are:

0. Make a guess for the critical value -- $w^0$
1. Compute the gradient of $L$ at $w^0$
2. Take a small step to $w^1 = w^0 - \alpha\nabla L\left(w^0\right)$
3. Compute the gradient of $L$ at $w^1$
4. Take a small step to $w^2 = w^1 - \alpha\nabla L\left(w^1\right)$
5. (repeat until the gradient gets close to $(0, ..., 0)$)

This $\alpha>0$ is a number that will be used in the algorithm as a multiplier of the steps the method will take. This is called the **learning rate**.

This idea seems plausible from the calculus ideas above because we just keep switching directions and making a step in the direction of the steepest downward path--the opposite direction as the gradient--until we reach a good place. This is a "greedy" algorithm because it just picks the quickest step in each iteration, which is fast, but it is likely to land in the first minimum it finds, which may or may not be optimal.

If you had two parameters, $L$ would be like a 3D curved surface. A nice visual to have in mind is a rain drop falling on a huge leaf. The droplet of water will move in the steepest downward direction due to gravity--but this direction *changes* as the drip follows the contours of the leaf. This is what gradient descent does.

Will the drip land in the physically lowest altitude part of the leaf? Maybe, but maybe not. If the rain drop lands on the edge, it will probably just roll off the edge. If the leaf has a few different "sinks," different initial locations of the rain drop might cause it to land in these different ones, some of which have lower altitudes than others. Now, if there is heavy rain and lots of rain drops land on the leaf, we can be pretty sure *some* of them will reach the lowest-altitude sink.

From this analogy, you might get the idea that we can make several initial guesses and run it to be more confident we will find the global minimum and not just a local minimum.

In the end, if some of our initial guesses are good choices, the step size $\alpha$ is not too big or too small, and the loss function is pretty well-behaved, the method will converge approximately to a local minimum.

#### Implementing Gradient Descent

Before writing code for gradient descent, let's import some libraries.

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.preprocessing import scale
import matplotlib.pyplot as plt
import pandas as pd

# increase the width of boxes in the notebook file (this is only cosmetic)
np.set_printoptions(linewidth=180)

Now, let's implement gradient descent and test it.

Gradient descent will need a few inputs:

* An explicit gradient function $
abla L(w)$
* A starting parameter vector $w^0$
* A learning rate $\alpha$
* A small positive value that we can use for a stopping condition for the derivative being sufficiently small (the tolerance)
* A maximum number of iterations

Here we compute each gradient analytically from its loss rather than approximating derivatives with finite differences.

In [ ]:
def gradient_descent(gradient_function, w, alpha, tolerance, max_iterations):
    w = np.array(w, dtype=float)

    for iteration in range(max_iterations):
        gradient = gradient_function(w)
        
        if np.linalg.norm(gradient) < tolerance:
            print('Gradient descent took', iteration, 'iterations to converge')
            print('The norm of the gradient is', np.linalg.norm(gradient))
            
            return w
        
        elif iteration == max_iterations - 1:
            print("Gradient descent failed")
            print('The gradient is', gradient)
            
            return w
        
        w -= alpha * gradient

Let's test it on some simple functions

In [ ]:
# Test L(w) = w^2.
loss = lambda w: w[0] ** 2
loss_gradient = lambda w: np.array([2 * w[0]])

w = gradient_descent(loss_gradient, w=[2], alpha=0.4, tolerance=1e-6, max_iterations=10000)
print(w, loss(w))

In [ ]:
# Test L(w) = sin(w).
loss = lambda w: np.sin(w[0])
loss_gradient = lambda w: np.array([np.cos(w[0])])

w = gradient_descent(loss_gradient, w=[2], alpha=0.5, tolerance=1e-4, max_iterations=10000)
print(w, loss(w))

### Linear Least Squares with Gradient Descent

Let's implement a class for linear regression using gradient descent to estimate $w$. For mean squared error, the explicit gradient is

$$\nabla L(w)=\frac{2}{n}X^T(Xw-y).$$

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import scale
import matplotlib.pyplot as plt
import pandas as pd

class LeastSquaresGradientDescent:
    def fit(self, X, y, w, alpha, tolerance, max_iterations):
        X = np.hstack((np.ones((X.shape[0], 1)), X))
        loss = lambda w: np.mean((X @ w - y) ** 2)
        loss_gradient = lambda w: (2 / len(y)) * X.T @ (X @ w - y)
        self.w = gradient_descent(
            loss_gradient, w, alpha, tolerance, max_iterations
        )
        self.loss = loss(self.w)
        return self

    def predict(self, X):
        X = np.hstack((np.ones((X.shape[0], 1)), X))
        return X @ self.w

Let's try it!

In [ ]:
X = np.array([[6], [7], [8], [9], [7]])
y = np.array([1, 2, 3, 3, 4])

# instantiate an least squares object, fit to data, predict data
model = LeastSquaresGradientDescent()

print('Fitting the model...\n')
model.fit(X, y, w=[0, 0], alpha=0.001, tolerance=0.01, max_iterations=100000)

y_pred = model.predict(X)

# print the predictions
print('\nThe predicted y values are', np.round(y_pred, 2))

# print the real y values
print('The real y values are', y)

# print the w values
w = model.w
print('The w values are', w)

# plot the training points
plt.scatter(X, y, label='Data')

# plot the fitted model with the data
x_plot = np.linspace(6, 10, 100)
y_plot = w[0] + w[1] * x_plot

# write a string for the formula
line_formula = 'y={:.3f}+{:.3f}x'.format(w[0], w[1])

# plot the model
plt.plot(x_plot, y_plot, 'r', label=line_formula)

# add a legend
plt.legend()

# return quality metrics
print('\nThe r^2 score is', r2_score(y, y_pred))
print('The mean squared error is', mean_squared_error(y, y_pred))
print('The mean absolute error is', mean_absolute_error(y, y_pred), '\n')

Clearly, the model seems to work. It's not a great fit, although the data is not really linear, so the model cannot fit it well.

### Cross-Validation: Train, Dev, and Test Datasets

(see the lecture notes: in Python, we can use the `train_test_split` function from the `scikit-learn` library to randomly assign datasets to training and testing sets.)

### Example: High School Graduate Rates in US States

Let's try to use least squares on a real dataset. The CSV file in `../../datasets/US_State_Data.csv` contains data from each U.S. state.

We would like to predict the output variable included, the high school graduation rate, from some input variables: including the crime rate (per 100,000 persons), the violent crime rate (per 100,000 persons), average teacher salary, student-to-teacher ratio, education expenditure per student, population density, and median household income.

This means we have 50 examples (one for each state), 7 input (predictor) variables, and one output (response) variable. In order to use the formula we derived above to attack the problem with least squares, we need to find the matrices $X$ and $y$.

In [ ]:
# import the data from the csv file to an numpy array
data = pd.read_csv('../../datasets/US_State_Data.csv', sep=',').to_numpy()

X = np.array(data[:,1:8], dtype=float)
y = np.array(data[:,8], dtype=float)

# split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1
)

X_train = scale(X_train)
X_test = scale(X_test)

# instantiate a least squares model
model = LeastSquaresGradientDescent()

# fit the model to the training data (find the w parameters)
print('Fitting the model...\n')
model.fit(X_train, y_train, w=np.zeros(X_train.shape[1] + 1), alpha=0.001,
          tolerance=0.01, max_iterations=100000)

# return the predicted outputs for the datapoints in the training set
y_train_pred = model.predict(X_train)

# print the coefficient of determination r^2
print('\nThe r^2 score is\t', r2_score(y_train, y_train_pred))

# print quality metrics
print('\nThe mean absolute error on the training set is\t', mean_absolute_error(y_train, y_train_pred))

# return the predicted outputs for the datapoints in the test set
y_test_pred = model.predict(X_test)

# print the predictions
print('\nThe predicted y values for the test set are\t', np.round(y_test_pred, 0))

# print the real y values
print('The real y values for the test set are\t', y_test)

# print the weights
print('\nThe weights are\t', model.w)

# print quality metrics
print('\nThe mean absolute error on the test set is\t', mean_absolute_error(y_test, y_test_pred), '\n')

### Comments on Gradient Descent

* We must be careful with the learning rate $\alpha$ and tolerance hyperparameters to help gradient descent converge.

* Each model supplies an explicit analytical gradient whose input is the parameter vector $w$.

* Exact gradients make each update proportional to $\nabla L(w)$ without repeatedly evaluating a finite-difference approximation.

* Backpropagation will let us compute these exact gradients efficiently for large neural networks.

* Gradient descent and related methods are the main driver of many machine learning problems that are based on to minimizing a loss function (least squares and neural networks, among others)

# Lecture 4: Classifiers, Activations, and Cross-Entropy


## Classification Losses and Activations


### Classification Problems

**Classification problems** are problems where we would like to take datapoints and assign them to an appropriate **class** based on some examples that have known classes.

### Examples

* If we have a dataset of medical records where each datapoint has a single patient's age, weight, blood pressure, status as a smoker or not, and other information and each patient is known to either have or not have kidney disease. We might want take data from a new patient and predict if he or she is likely to develop kidney disease.

* If we have a dataset of labeled images of cat and dogs, we might want to take a new image and classify whether it has a dog or a cat. (How does Google image search know how to find pictures of what you search?)

* If we have a dataset of audio files, each of which is a jazz, classical, rock, pop, or hip-hop song, we might want to predict the genre of a new audio file. (Spotify does this kind of analysis to recommend songs based on your listening history.)

* If we have a dataset of traffic logs on a network, some known to be infected by a specific virus and some are not, we might want to use this information to classify a new traffic log as likely to be infected or not.

* If we have a dataset of sounds of people speaking along with transcripts of the words, we might want to classify the words spoken into a microphone. (Think Siri!)

In all of these cases, the information can be represented as a point in the $n$-dimensional real space.

* The medical records would have numbers for age, weight, and blood pressure and a binary digit for non-smoker or smoker.

![medical_records](../images/medicalRecords.png)

* The dog/cat images might have three channels for a picture, meaning three numbers for each pixel (the red, green, and blue levels) like the bird picture below.

![bird_rgb](../images/birdRGB.png)

* The audio files might have numbers specifying the type of sound for the song many times per second.

![audio_img](../images/audioFile.png)

* The network traffic logs might have numbers of packets transferred, file size, ports, addresses, the content of the packets, etc.

* The audio files might have numbers specifying the type of sound for a word many times per second.

![audio_speech](../images/audioWords.png)

The blue lines indicate the beginning of a word and the red lines indicate the ends of words.

In all mature applications, there are likely preprocessing steps done before the classification is done.

I chose these applications to demonstrate two things: (1) classification problems are interesting and useful in almost every area of study and (2) a huge class of classification problems have much in common mathematically. All apply to datapoints, although some types of data may have far more dimensions than others--a medical record may only have 10 to 12 numbers, but a 12-megapixel photo from the latest iPhone would have $4,000\times 3,000\times 3=36,000,000$ numbers, 3 for each pixel).

### The Math of a Classification Problem

To exploit the similarities, let's abstract away the specifics of the applications for now and think about how to describe a classification problem mathematically. Consider a $d$-dimensional point, or vector, $x_1\in\mathbb{R}^d$ and denote $x=(x_{11},x_{12},...,x_{1d})$. $x$ is a member one of $k$ classes $C=\{c_1, c_2, ..., c_k\}$. We call the point $x_1$ an **example** and we call the class the **label** of $x_1$.

The goal of a classification problem is to find a function $M:\mathbb{R}^d\to C$ mapping each example $x_1$ to its class $y_1=M(x_1)$ and will generalize to successfully classify new, unlabeled datapoints with high accuracy.

This will segment the space $\mathbb{R}^d$ into sets $X_j=\{x_1\in\mathbb{R}^d | M(x_1)=c_j\}$ corresponding to each class. In the image below, for example, the space $\mathbb{R}^2$ is partitioned into three sets colored red, blue, and green.

![knn_tess](../images/knnTessellation.png)

The colored points are labeled examples and the $\mathbb{R}^2$ space is colored by the class to be assigned to points in different regions. (image from Wikipedia)

### Classification Algorithms

There are many algorithms used for classification. Some of the most popular include

* Bayes and naive Bayes
* $k$-nearest neighbors
* decision trees
* logistic regression
* support vector machines
* neural networks (many types)

as well as tree-based algorithms that systematically apply an ensemble of different classifiers. Any of the methods above are good choices and there are pros and cons of each, but we only consider tiny neural networks and logistic regression this week.

An array of classifiers are used on a few datasets in the code below from [scikit-learn](https://scikit-learn.org/stable/auto_examples/classification/plot_classifier_comparison.html). I ask you not to focus on the code, but look at the diagram it generates. The diagram shows how different classifiers come to quite different results at classifying 2D points into the red and blue classes.

### Diversity of Classifiers

There are many types of classifiers, all of which break down the space differently due to their different approaches.

In [ ]:
print(__doc__)


# Code source: Gaël Varoquaux
#              Andreas Müller
# Modified for documentation by Jaques Grobler
# License: BSD 3 clause

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

mesh_step = 0.02

names = ["Nearest Neighbors", "Linear SVM", "RBF SVM", "Gaussian Process",
         "Decision Tree", "Random Forest", "Neural Net", "AdaBoost",
         "Naive Bayes", "QDA"]

classifiers = [
    KNeighborsClassifier(3),
    SVC(kernel="linear", C=0.025),
    SVC(gamma=2, C=1),
    GaussianProcessClassifier(1.0 * RBF(1.0)),
    DecisionTreeClassifier(max_depth=5),
    RandomForestClassifier(max_depth=5, n_estimators=10, max_features=1),
    MLPClassifier(alpha=1, max_iter=1000),
    AdaBoostClassifier(),
    GaussianNB(),
    QuadraticDiscriminantAnalysis()]

X, y = make_classification(n_features=2, n_redundant=0, n_informative=2,
                           random_state=1, n_clusters_per_class=1)
rng = np.random.RandomState(2)
X += 2 * rng.uniform(size=X.shape)
linearly_separable = (X, y)

datasets = [make_moons(noise=0.3, random_state=0),
            make_circles(noise=0.2, factor=0.5, random_state=1),
            linearly_separable
            ]

figure = plt.figure(figsize=(27, 9))
i = 1
# iterate over datasets
for ds_cnt, ds in enumerate(datasets):
    # preprocess dataset, split into training and test part
    X, y = ds
    X = StandardScaler().fit_transform(X)
    X_train, X_test, y_train, y_test = \
        train_test_split(X, y, test_size=.4, random_state=42)

    x_min, x_max = X[:, 0].min() - .5, X[:, 0].max() + .5
    y_min, y_max = X[:, 1].min() - .5, X[:, 1].max() + .5
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, mesh_step),
        np.arange(y_min, y_max, mesh_step),
    )

    # just plot the dataset first
    cm = plt.cm.RdBu
    cm_bright = ListedColormap(['#FF0000', '#0000FF'])
    ax = plt.subplot(len(datasets), len(classifiers) + 1, i)
    if ds_cnt == 0:
        ax.set_title("Input data")
    # Plot the training points
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright,
               edgecolors='k')
    # Plot the testing points
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_bright, alpha=0.6,
               edgecolors='k')
    ax.set_xlim(xx.min(), xx.max())
    ax.set_ylim(yy.min(), yy.max())
    ax.set_xticks(())
    ax.set_yticks(())
    i += 1

    # iterate over classifiers
    for name, clf in zip(names, classifiers):
        ax = plt.subplot(len(datasets), len(classifiers) + 1, i)
        clf.fit(X_train, y_train)
        score = clf.score(X_test, y_test)

        # Plot the decision boundary. For that, we will assign a color to each
        # point in the mesh [x_min, x_max]x[y_min, y_max].
        if hasattr(clf, "decision_function"):
            z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
        else:
            z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]

        # Put the result into a color plot
        z = z.reshape(xx.shape)
        ax.contourf(xx, yy, z, cmap=cm, alpha=.8)

        # Plot the training points
        ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright,
                   edgecolors='k')
        # Plot the testing points
        ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_bright,
                   edgecolors='k', alpha=0.6)

        ax.set_xlim(xx.min(), xx.max())
        ax.set_ylim(yy.min(), yy.max())
        ax.set_xticks(())
        ax.set_yticks(())
        if ds_cnt == 0:
            ax.set_title(name)
        ax.text(xx.max() - .3, yy.min() + .3, ('%.2f' % score).lstrip('0'),
                size=15, horizontalalignment='right')
        i += 1

plt.tight_layout()
plt.show()

### Binary Logistic Classification

First, let's import some libraries.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sn

from scipy.stats import multivariate_normal
from scipy.special import expit
from sklearn import datasets
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import scale

For binary logistic classification, the model probability is $p=\sigma(Xw)$. The binary cross-entropy loss and its explicit gradient are

$$L(w)=-\frac{1}{n}\sum_i\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right],$$

$$\nabla L(w)=\frac{1}{n}X^T(p-y).$$

In [ ]:
class LogisticClassifierGradientDescent:
    def fit(self, X, y, w, alpha, tolerance, max_iterations):
        self.x_mean = X.mean(axis=0)
        self.x_std = X.std(axis=0)
        self.x_std[self.x_std == 0] = 1

        X = (X - self.x_mean) / self.x_std
        X = np.hstack((np.ones((X.shape[0], 1)), X))

        def loss(w):
            p = np.clip(expit(X @ w), 1e-12, 1 - 1e-12)
            return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

        def loss_gradient(w):
            p = expit(X @ w)
            return (1 / len(y)) * X.T @ (p - y)

        self.w = gradient_descent(
            loss_gradient, w, alpha, tolerance, max_iterations
        )
        self.loss = loss(self.w)
        return self

    def predict_proba(self, X):
        X = (X - self.x_mean) / self.x_std
        X = np.hstack((np.ones((X.shape[0], 1)), X))
        return expit(X @ self.w)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

### Breast Cancer Classification

In [ ]:
# read the breast cancer dataset
breast_cancer_data = datasets.load_breast_cancer()

# find the data and labels
X = breast_cancer_data.data
y = breast_cancer_data.target

# do a train/test split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1
)

# build the logistic classifier
model = LogisticClassifierGradientDescent()

# Fit the logistic classifier to the training data.
model.fit(X_train, y_train,
          w=np.zeros(X.shape[1] + 1), alpha=0.1,
          tolerance=0.001, max_iterations=10000)

# predict the Labels of the training set 
y_train_pred = model.predict(X_train)

# print quality metrics 
print(f'\nTrain Classification Report: \n {classification_report(y_train, y_train_pred)}')

# predict the Labels of the test set 
y_test_pred = model.predict(X_test)

# print quality metrics 
print(f'Test Classification Report: \n {classification_report(y_test, y_test_pred)}')

print('Test Confusion Matrix: \n')

sn.heatmap(confusion_matrix(y_test, y_test_pred), annot=True)